In [2]:
# Build Mortality Sheet — basis_master
#Extracts every age×gender×rate table from `Basis and Assumptions.xlsx`
#and writes them to:
#1. `basis_master_v2026_07.xlsx` (sheet: `mortality`)
#2. `basis_v2026_07/mortality.parquet`

In [3]:
from __future__ import annotations
from pathlib import Path
import polars as pl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo

# ============================================================
# PATHS
# ============================================================
BASE = Path(
    r"J:\Product & Client Solutions\Toolbox\Calculators"
    r"\Business Tools\Tender Summary\GAP Pricing Engine"
)

SOURCE       = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX  = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR = BASE / "basis_snapshots" / "basis_v2026_07"

BASIS_VERSION = "B2026_07"

# Make sure output folders exist
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Source        : {SOURCE}")
print(f"📄 Master xlsx   : {MASTER_XLSX}")
print(f"📦 Snapshot dir  : {SNAPSHOT_DIR}")
print(f"🏷️  Basis version : {BASIS_VERSION}")
assert SOURCE.exists(), f"Source not found: {SOURCE}"
print("\n✅ Setup ready")

📂 Source        : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\Basis and Assumptions.xlsx
📄 Master xlsx   : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_master\basis_master_v2026_07.xlsx
📦 Snapshot dir  : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07
🏷️  Basis version : B2026_07

✅ Setup ready


In [4]:
def find_anchors(ws, text: str) -> list[tuple[int, int]]:
    """Return list of (row, col) 1-indexed for cells exactly matching text."""
    hits = []
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == text:
                hits.append((cell.row, cell.column))
    return hits


def extract_mortality(ws, anchor_row: int, anchor_col: int,
                      first_age: int, last_age: int,
                      m_offset: int = 1, f_offset: int = 2
                      ) -> list[tuple[int, float | None, float | None]]:
    """Walk downward from anchor extracting (age, M rate, F rate) rows.

    m_offset / f_offset are column offsets from the anchor cell.
    Defaults preserve existing behaviour (M in col+1, F in col+2).
    """
    rows = []
    r = anchor_row + 1
    max_iters = last_age - first_age + 10
    for _ in range(max_iters):
        age_v = ws.cell(row=r, column=anchor_col).value
        if age_v is None:
            break
        try:
            age = int(age_v)
        except (ValueError, TypeError):
            break
        m_v = ws.cell(row=r, column=anchor_col + m_offset).value
        f_v = ws.cell(row=r, column=anchor_col + f_offset).value
        rows.append((
            age,
            float(m_v) if m_v is not None else None,
            float(f_v) if f_v is not None else None,
        ))
        r += 1
    return rows
print("✅ Helpers ready")

✅ Helpers ready


In [5]:
SPECS = [
    # -------- GLA + LSDB + Spouse + ADB/AccDB/Dism --------
    dict(product="gla",         sheet="GLA Basis",
         anchor="Age Next Birthday", occurrence=1,
         first=18, last=99, age_def="ANB"),
    dict(product="lsdb",        sheet="LSDB Basis",
         anchor="Age Next Birthday", occurrence=1,
         first=18, last=65, age_def="ANB"),
    dict(product="spouse",      sheet="Spouse Basis",
         anchor="ANB", occurrence=1,
         first=16, last=65, age_def="ANB"),
    dict(product="spouse_aids", sheet="Spouse Basis",
         anchor="ANB", occurrence=2,
         first=16, last=65, age_def="ANB"),
    dict(product="adb",         sheet="ADB Basis",
         anchor="Age Next", occurrence=1,
         first=17, last=90, age_def="ANB",
         m_offset=3, f_offset=4),
    dict(product="accdb",       sheet="AccDB Basis",
         anchor="ANB", occurrence=1,
         first=16, last=65, age_def="ANB"),
    dict(product="dism",        sheet="Dismemberment Basis",
         anchor="ANB", occurrence=1,
         first=16, last=65, age_def="ANB"),

    # -------- CI Old List (6 side-by-side) --------
    dict(product="ci_core_acc",           sheet="CI Basis",
         anchor="ANB", occurrence=1, first=15, last=70, age_def="ANB"),
    dict(product="ci_comprehensive_acc",  sheet="CI Basis",
         anchor="ANB", occurrence=2, first=15, last=70, age_def="ANB"),
    dict(product="ci_extensive_acc",      sheet="CI Basis",
         anchor="ANB", occurrence=3, first=15, last=70, age_def="ANB"),
    dict(product="ci_core_std",           sheet="CI Basis",
         anchor="ANB", occurrence=4, first=15, last=70, age_def="ANB"),
    dict(product="ci_comprehensive_std",  sheet="CI Basis",
         anchor="ANB", occurrence=5, first=15, last=70, age_def="ANB"),
    dict(product="ci_extensive_std",      sheet="CI Basis",
         anchor="ANB", occurrence=6, first=15, last=70, age_def="ANB"),

    # -------- CI AIDS + GLA AIDS (further right on CI Basis) --------
    dict(product="gla_aids", sheet="CI Basis",
         anchor="ANB", occurrence=7, first=16, last=89, age_def="ANB"),
    dict(product="ci_aids_acc",  sheet="CI Basis",          # ← was ci_aids
         anchor="ANB", occurrence=8, first=16, last=89, age_def="ANB"),

    # -------- CI New List (ALB, all Accelerators) --------
    dict(product="ci_cancer_protect_acc",     sheet="CI Basis",    # ← + _acc
         anchor="Age", occurrence=1, first=18, last=65, age_def="ALB"),
    dict(product="ci_essential_acc",          sheet="CI Basis",    # ← + _acc
         anchor="Age", occurrence=2, first=18, last=65, age_def="ALB"),
    dict(product="ci_comprehensive_plus_acc", sheet="CI Basis",    # ← + _acc
         anchor="Age", occurrence=3, first=18, last=65, age_def="ALB"),
    dict(product="ci_extensive_plus_acc",     sheet="CI Basis",    # ← + _acc
         anchor="Age", occurrence=4, first=18, last=65, age_def="ALB"),
]

print(f"✅ {len(SPECS)} table specs defined")

✅ 19 table specs defined


In [6]:
print(f"📂 Opening source: {SOURCE.name}\n")
wb = openpyxl.load_workbook(SOURCE, data_only=True)

all_rows = []
print(f"🔎 Extracting {len(SPECS)} mortality tables...\n")

for spec in SPECS:
    ws = wb[spec["sheet"]]
    anchors = find_anchors(ws, spec["anchor"])
    occ = spec["occurrence"] - 1

    if occ >= len(anchors):
        print(f"  ❌ {spec['product']:<25} anchor '{spec['anchor']}' "
              f"#{spec['occurrence']} NOT FOUND ({len(anchors)} matches)")
        continue

    r, c = anchors[occ]
    rows = extract_mortality(
        ws, r, c,
        spec["first"], spec["last"],
        m_offset=spec.get("m_offset", 1),
        f_offset=spec.get("f_offset", 2),
)

    if not rows:
        print(f"  ❌ {spec['product']:<25} no data extracted")
        continue

    first_age, last_age = rows[0][0], rows[-1][0]
    warn = ""
    if first_age != spec["first"] or last_age != spec["last"]:
        warn = f"  ⚠️  ages {first_age}-{last_age} (expected {spec['first']}-{spec['last']})"

    print(f"  ✅ {spec['product']:<25} #{spec['occurrence']} → {len(rows):>3} rows{warn}")

    for age, m, f in rows:
        if m is not None:
            all_rows.append(dict(product=spec["product"], age_def=spec["age_def"],
                                 age=age, gender="M", rate=m))
        if f is not None:
            all_rows.append(dict(product=spec["product"], age_def=spec["age_def"],
                                 age=age, gender="F", rate=f))

df = pl.DataFrame(all_rows).sort(["product", "age", "gender"])

print(f"\n📊 Total rows: {df.height}  |  Distinct products: {df['product'].n_unique()}")
df.group_by("product").agg(pl.len().alias("n")).sort("product")

📂 Opening source: Basis and Assumptions.xlsx

🔎 Extracting 19 mortality tables...

  ✅ gla                       #1 →  82 rows
  ✅ lsdb                      #1 →  48 rows
  ✅ spouse                    #1 →  50 rows
  ✅ spouse_aids               #2 →  50 rows
  ✅ adb                       #1 →  74 rows
  ✅ accdb                     #1 →  50 rows
  ✅ dism                      #1 →  50 rows
  ✅ ci_core_acc               #1 →  56 rows
  ✅ ci_comprehensive_acc      #2 →  56 rows
  ✅ ci_extensive_acc          #3 →  56 rows
  ✅ ci_core_std               #4 →  56 rows
  ✅ ci_comprehensive_std      #5 →  56 rows
  ✅ ci_extensive_std          #6 →  56 rows
  ✅ gla_aids                  #7 →  74 rows
  ✅ ci_aids_acc               #8 →  74 rows
  ✅ ci_cancer_protect_acc     #1 →  48 rows
  ✅ ci_essential_acc          #2 →  48 rows
  ✅ ci_comprehensive_plus_acc #3 →  48 rows
  ✅ ci_extensive_plus_acc     #4 →  48 rows

📊 Total rows: 2160  |  Distinct products: 19


product,n
str,u32
"""accdb""",100
"""adb""",148
"""ci_aids_acc""",148
"""ci_cancer_protect_acc""",96
"""ci_comprehensive_acc""",112
…,…
"""gla""",164
"""gla_aids""",148
"""lsdb""",96


In [7]:
# Three key sanity values. If any of these FAIL, something's wrong.
SANITY_CHECKS = [
    ("gla",                  25, "M", 1.7911126833),
    ("ci_comprehensive_acc", 40, "F", 3.4075),
    ("lsdb",                 50, "F", 2.6631241108),
    # NEW — adb sanity anchors
    ("adb", 17, "M", 0.91574308398177),
    ("adb", 17, "F", 0.16688110610400145),
    ("adb", 40, "M", 0.9532724458142132),
    ("adb", 40, "F", 0.20492436932053687),
    ("adb", 90, "M", 6.516631880442201),
    ("adb", 90, "F", 3.343655988874499),
]

print("🧪 Sanity checks:\n")
for product, age, gender, expected in SANITY_CHECKS:
    actual = (df.filter(
        (pl.col("product") == product) &
        (pl.col("age") == age) &
        (pl.col("gender") == gender)
    )["rate"].to_list())
    
    if not actual:
        status = "❌ NOT FOUND"
        actual_str = "-"
    else:
        actual_val = actual[0]
        match = abs(actual_val - expected) < 1e-6
        status = "✅ MATCH" if match else "❌ MISMATCH"
        actual_str = f"{actual_val:.10f}"
    
    print(f"  {status}  {product:<25} age={age} {gender}  "
          f"expected={expected:.10f}  actual={actual_str}")

🧪 Sanity checks:

  ✅ MATCH  gla                       age=25 M  expected=1.7911126833  actual=1.7911126833
  ✅ MATCH  ci_comprehensive_acc      age=40 F  expected=3.4075000000  actual=3.4075000000
  ✅ MATCH  lsdb                      age=50 F  expected=2.6631241108  actual=2.6631241108
  ✅ MATCH  adb                       age=17 M  expected=0.9157430840  actual=0.9157430840
  ✅ MATCH  adb                       age=17 F  expected=0.1668811061  actual=0.1668811061
  ✅ MATCH  adb                       age=40 M  expected=0.9532724458  actual=0.9532724458
  ✅ MATCH  adb                       age=40 F  expected=0.2049243693  actual=0.2049243693
  ✅ MATCH  adb                       age=90 M  expected=6.5166318804  actual=6.5166318804
  ✅ MATCH  adb                       age=90 F  expected=3.3436559889  actual=3.3436559889


In [8]:
# ---------- 1. Write parquet snapshot ----------
parquet_path = SNAPSHOT_DIR / "mortality.parquet"
df.write_parquet(parquet_path)
print(f"📦 Parquet snapshot: {parquet_path}")
print(f"   Size: {parquet_path.stat().st_size / 1024:.1f} KB")

# ---------- 2. Write to master xlsx ----------
if MASTER_XLSX.exists():
    wb_out = openpyxl.load_workbook(MASTER_XLSX)
    if "mortality" in wb_out.sheetnames:
        del wb_out["mortality"]
else:
    wb_out = openpyxl.Workbook()
    if "Sheet" in wb_out.sheetnames:
        del wb_out["Sheet"]

ws = wb_out.create_sheet("mortality")

# Header
headers = ["product", "age_def", "age", "gender", "rate"]
ws.append(headers)
for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

# Data
for row in df.iter_rows():
    ws.append(row)

# Formatting
widths = {"A": 26, "B": 10, "C": 8, "D": 10, "E": 22}
for col, w in widths.items():
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A2"

# Excel Table (autofilter + banded rows)
ref = f"A1:E{ws.max_row}"
table = Table(displayName="tbl_mortality", ref=ref)
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws.add_table(table)

wb_out.save(MASTER_XLSX)
print(f"\n📄 Master xlsx:      {MASTER_XLSX}")
print(f"   Sheet 'mortality': {df.height} rows written")

print("\n🎉 Done!")

📦 Parquet snapshot: J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\mortality.parquet
   Size: 14.3 KB

📄 Master xlsx:      J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_master\basis_master_v2026_07.xlsx
   Sheet 'mortality': 2160 rows written

🎉 Done!
